# 📊 QVT スコア予測率検証 & Optuna 最適化

```
各CSVのQVTスコア
    ↓  +252営業日後の終値をyfinanceから取得
予測率（騰落方向の一致率）を計算
    ↓  Optuna
QVT重み / Q・V・T各サブスコアのパラメータを最適化
```

## 確認済みフォーミュラ（逆算結果）
```
q_score  = q1 * 0.5  + q3 * 0.5
v_score  = v1 * 0.45 + v2 * 0.30 + v3 * 0.25
qvt_score = t_score * 0.35 + q_score * 0.35 + v_score * 0.30
```

**使い方**: セルを上から順に `Shift+Enter` で実行

In [ ]:
# ─── セル1: パス設定 ──────────────────────────────────────────────────────────

import os

# ★ CSVファイルのディレクトリを指定
CSV_DIR = r'.'   # カレントディレクトリ or フルパスに変更

# ★ 対象ファイル（同ディレクトリにある場合はファイル名のみ）
CSV_FILES = [
    '7203T.csv',   # トヨタ
    '3864T.csv',   # 三菱製紙
    '5333T.csv',   # NGKインサレーター
    '7932T.csv',   # ニッピ
    '8113T.csv',   # ユニ・チャーム
]

# ★ Optuna 試行回数
N_TRIALS = 100

# ★ QVTスコアの「買いシグナル閾値」
# 　→ 最適化でも探索するが初期値として使用
QVT_THRESHOLD = 45.0

print('✅ パス設定完了')
for f in CSV_FILES:
    path = os.path.join(CSV_DIR, f)
    print(f'   {f}: {"✅" if os.path.exists(path) else "❌ 見つかりません"}')

In [ ]:
# ─── セル2: ライブラリ & インポート ───────────────────────────────────────────

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['optuna', 'yfinance', 'pandas', 'numpy', 'plotly', 'ipywidgets']:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        install(pkg)

import warnings
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

THEME = dict(
    paper_bgcolor='#0a0e1a',
    plot_bgcolor='#111827',
    font=dict(color='#94a3b8', family='sans-serif'),
)
print('✅ 全ライブラリ読み込み完了')

In [ ]:
# ─── セル3: CSV読み込み & データ確認 ─────────────────────────────────────────

def load_csv(filename):
    """CSVを読み込み、数値変換・欠損除去を行う"""
    path = os.path.join(CSV_DIR, filename)
    df = pd.read_csv(path, encoding='utf-8-sig')  # BOM対応
    df = df.replace('-', np.nan)
    numeric_cols = [
        'close', 'ma_25', 'ma_50', 'rsi', 'high_52w', 'low_52w',
        'per', 'pbr', 'q_score', 'q1', 'q3',
        'v_score', 'v1', 'v2', 'v3',
        't_score', 'qvt_score',
        'roe', 'roa', 'equity_ratio', 'operating_margin',
        'de_ratio', 'interest_coverage', 'ev_ebitda',
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    return df

# 全CSV読み込み
raw_data = {}
for f in CSV_FILES:
    ticker = f.replace('.csv', '').replace('T', '') + '.T'
    df = load_csv(f)
    valid = df.dropna(subset=['close', 'qvt_score', 'q_score', 'v_score', 't_score',
                               'q1', 'q3', 'v1', 'v2', 'v3'])
    raw_data[ticker] = valid.reset_index(drop=True)
    print(f'{ticker}: {len(valid)}行有効 | {valid["date"].min().date()} ～ {valid["date"].max().date()}')
    print(f'  QVT範囲: {valid["qvt_score"].min():.1f} ～ {valid["qvt_score"].max():.1f}')

print(f'\n✅ {len(raw_data)}銘柄 読み込み完了')

In [ ]:
# ─── セル4: +252営業日後の終値をyfinanceから取得 ─────────────────────────────
# 「252日後が休場日なら253日後（または最初の営業日）の終値」を使用

import time

_price_cache = {}  # yfinance取得コスト削減用キャッシュ

def get_future_price(ticker: str, base_date: pd.Timestamp,
                     n_bdays: int = 252, max_offset: int = 5) -> float:
    """
    base_date から n_bdays 営業日後の終値を返す。
    休場日なら最大 max_offset 日先まで探索。
    キャッシュ済みの価格系列から高速に返す。
    """
    if ticker not in _price_cache:
        return np.nan
    prices = _price_cache[ticker]  # pd.Series(index=DatetimeIndex)

    target = base_date + pd.tseries.offsets.BDay(n_bdays)
    for delta in range(max_offset + 1):
        candidate = target + pd.tseries.offsets.BDay(delta)
        if candidate in prices.index:
            return float(prices[candidate])
    return np.nan


def fetch_price_series(ticker: str,
                       start: str, end: str) -> pd.Series:
    """yfinanceから終値系列をfetch（キャッシュに保存）"""
    if ticker in _price_cache:
        return _price_cache[ticker]
    raw = yf.download(ticker, start=start, end=end,
                      progress=False, auto_adjust=True)
    if raw.empty:
        return pd.Series(dtype=float)
    raw.columns = [c[0] if isinstance(c, tuple) else c for c in raw.columns]
    prices = raw['Close']
    prices.index = pd.to_datetime(prices.index).normalize()
    _price_cache[ticker] = prices
    return prices


# ── 全銘柄の価格データを一括取得
print('📡 yfinanceから価格データを取得中...')
for ticker, df in raw_data.items():
    start_str = df['date'].min().strftime('%Y-%m-%d')
    # +252営業日分余裕をみて2年後まで取得
    end_ts = df['date'].max() + pd.tseries.offsets.BDay(260)
    end_str = end_ts.strftime('%Y-%m-%d')
    prices = fetch_price_series(ticker, start_str, end_str)
    print(f'  {ticker}: {len(prices)}日分 ({prices.index.min().date()} ～ {prices.index.max().date()})')
    time.sleep(0.3)  # レート制限対策

print('\n✅ 価格データ取得完了')

In [ ]:
# ─── セル5: QVTスコア予測率の計算 ────────────────────────────────────────────
#
# 予測率の定義:
#   「QVTスコアが閾値以上の日」に買い → 252営業日後の終値が上昇していれば的中
#   予測率 = 的中数 / シグナル発生数
#
# また、スコア四分位ごとの予測率も集計（スコアと騰落の相関確認）

def calc_qvt_score(row, wt, wq, wv,
                   wq1, wq3,
                   wv1, wv2, wv3):
    """カスタム重みでQVTスコアを再計算"""
    q = wq1 * row['q1'] + wq3 * row['q3']
    v = wv1 * row['v1'] + wv2 * row['v2'] + wv3 * row['v3']
    t = row['t_score']
    total = wt + wq + wv
    if total == 0:
        return 50.0
    return (wt * t + wq * q + wv * v) / total


def build_prediction_df(ticker: str, df: pd.DataFrame,
                        wt=0.35, wq=0.35, wv=0.30,
                        wq1=0.5, wq3=0.5,
                        wv1=0.45, wv2=0.30, wv3=0.25,
                        threshold=45.0, n_bdays=252) -> pd.DataFrame:
    """
    各行に対して:
      - カスタムQVTスコアを計算
      - +n_bdays後の終値を取得
      - 上昇/下落フラグを付与
    """
    rows = []
    for _, row in df.iterrows():
        qvt = calc_qvt_score(row, wt, wq, wv, wq1, wq3, wv1, wv2, wv3)
        future_price = get_future_price(ticker, row['date'], n_bdays=n_bdays)
        if np.isnan(future_price) or np.isnan(row['close']):
            continue
        ret_1y = (future_price - row['close']) / row['close']
        rows.append({
            'date':         row['date'],
            'close':        row['close'],
            'future_price': future_price,
            'return_1y':    ret_1y,
            'rose':         ret_1y > 0,
            'qvt_score':    qvt,
            'signal':       qvt >= threshold,
            'orig_qvt':     row['qvt_score'],
            'q_score':      row['q_score'],
            'v_score':      row['v_score'],
            't_score':      row['t_score'],
        })
    return pd.DataFrame(rows)


def calc_accuracy(pred_df: pd.DataFrame, threshold: float) -> dict:
    """予測率・騰落別パフォーマンスを計算"""
    sigs = pred_df[pred_df['signal']]
    if len(sigs) == 0:
        return {'accuracy': 0, 'n_signal': 0, 'avg_return': 0}
    accuracy  = sigs['rose'].mean()
    avg_ret   = sigs['return_1y'].mean()
    return {
        'accuracy':   accuracy,
        'n_signal':   len(sigs),
        'avg_return': avg_ret,
        'sharpe':     sigs['return_1y'].mean() / (sigs['return_1y'].std() + 1e-9),
        'best_ret':   sigs['return_1y'].max(),
        'worst_ret':  sigs['return_1y'].min(),
    }


# ── デフォルト重みで予測率を計算
print('📊 デフォルト重み（T=0.35, Q=0.35, V=0.30）で予測率計算中...')
print(f'   閾値 = {QVT_THRESHOLD}')
print()

default_preds = {}
summary_rows  = []

for ticker, df in raw_data.items():
    pred_df = build_prediction_df(
        ticker, df, threshold=QVT_THRESHOLD
    )
    default_preds[ticker] = pred_df
    metrics = calc_accuracy(pred_df, QVT_THRESHOLD)
    all_acc = pred_df['rose'].mean() if len(pred_df) > 0 else 0
    summary_rows.append({
        'ticker':      ticker,
        '全行騰落率':   f'{all_acc*100:.1f}%',
        'シグナル数':   metrics['n_signal'],
        '予測率':       f'{metrics["accuracy"]*100:.1f}%',
        '平均1年リターン': f'{metrics["avg_return"]*100:.1f}%',
        '最良':         f'{metrics["best_ret"]*100:.1f}%',
        '最悪':         f'{metrics["worst_ret"]*100:.1f}%',
        '検証可能行数':  len(pred_df),
    })
    print(f'{ticker}: 検証行={len(pred_df)}, シグナル={metrics["n_signal"]}, '
          f'予測率={metrics["accuracy"]*100:.1f}%, '
          f'平均1年リターン={metrics["avg_return"]*100:.1f}%')

print()
print('=== サマリーテーブル ===')
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

In [ ]:
# ─── セル6: QVTスコア四分位ごとの予測率チャート ──────────────────────────────

all_pred = pd.concat(
    [df.assign(ticker=t) for t, df in default_preds.items()],
    ignore_index=True
)

if len(all_pred) > 0:
    # QVTスコアをビン分割（10点刻み）
    bins = list(range(0, 110, 10))
    all_pred['score_bin'] = pd.cut(all_pred['qvt_score'], bins=bins,
                                    labels=[f'{b}-{b+10}' for b in bins[:-1]])
    bin_stats = all_pred.groupby('score_bin', observed=True).agg(
        count=('rose', 'count'),
        accuracy=('rose', 'mean'),
        avg_ret=('return_1y', 'mean'),
    ).reset_index()

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('QVTスコア帯ごとの騰落予測率', '平均1年リターン'),
    )
    colors = ['#ef4444' if a < 0.5 else '#10b981' for a in bin_stats['accuracy']]

    fig.add_trace(go.Bar(
        x=bin_stats['score_bin'].astype(str),
        y=bin_stats['accuracy'] * 100,
        name='予測率(%)', marker_color=colors,
        text=bin_stats['count'].apply(lambda x: f'n={x}'),
        textposition='outside',
    ), row=1, col=1)
    fig.add_hline(y=50, line=dict(color='#f59e0b', dash='dash'), row=1, col=1)

    fig.add_trace(go.Bar(
        x=bin_stats['score_bin'].astype(str),
        y=bin_stats['avg_ret'] * 100,
        name='平均リターン(%)',
        marker_color=['#ef4444' if r < 0 else '#3b82f6' for r in bin_stats['avg_ret']],
    ), row=1, col=2)
    fig.add_hline(y=0, line=dict(color='#f59e0b', dash='dash'), row=1, col=2)

    fig.update_layout(
        **THEME, height=420,
        title=dict(text='📈 QVTスコア帯ごとの1年後騰落分析（デフォルト重み）',
                   font=dict(size=15, color='#f1f5f9')),
        showlegend=False,
        margin=dict(l=0, r=0, t=60, b=0),
    )
    fig.update_xaxes(gridcolor='#1e293b', tickangle=45)
    fig.update_yaxes(gridcolor='#1e293b', zeroline=True, zerolinecolor='#334155')
    fig.show()
else:
    print('⚠️ 検証可能データがありません（252日後価格が未取得の可能性）')

In [ ]:
# ─── セル7: 銘柄別スコア時系列 & 1年後リターン散布図 ─────────────────────────

ticker_list = list(default_preds.keys())
fig = make_subplots(
    rows=len(ticker_list), cols=1,
    shared_xaxes=False,
    subplot_titles=[f'{t} — QVTスコア vs 1年後リターン' for t in ticker_list],
    vertical_spacing=0.08,
)

for i, (ticker, pred_df) in enumerate(default_preds.items(), 1):
    if len(pred_df) == 0:
        continue
    colors = ['#10b981' if r else '#ef4444' for r in pred_df['rose']]
    fig.add_trace(go.Scatter(
        x=pred_df['qvt_score'],
        y=pred_df['return_1y'] * 100,
        mode='markers',
        marker=dict(color=colors, size=6, opacity=0.7),
        name=ticker,
        text=pred_df['date'].dt.strftime('%Y-%m-%d'),
        hovertemplate=(
            'QVT: %{x:.1f}<br>'
            '1年後: %{y:.1f}%<br>'
            '日付: %{text}'
        ),
    ), row=i, col=1)
    # 閾値縦線
    fig.add_vline(x=QVT_THRESHOLD,
                  line=dict(color='#f59e0b', dash='dash', width=1),
                  row=i, col=1)
    fig.add_hline(y=0,
                  line=dict(color='#6b7280', dash='dot', width=0.8),
                  row=i, col=1)

fig.update_layout(
    **THEME,
    height=280 * len(ticker_list),
    title=dict(text='🔵=上昇 🔴=下落  黄破線=閾値',
               font=dict(size=13, color='#94a3b8')),
    showlegend=False,
    margin=dict(l=0, r=0, t=60, b=0),
)
fig.update_xaxes(gridcolor='#1e293b', title_text='QVTスコア')
fig.update_yaxes(gridcolor='#1e293b', zeroline=True,
                 zerolinecolor='#334155', title_text='1年後リターン(%)')
fig.show()

In [ ]:
# ─── セル8: Optuna 最適化 ─────────────────────────────────────────────────────
#
# 最適化する変数:
#   【QVT合成重み】 wt, wq, wv  （合計1に正規化）
#   【Qスコア内部重み】 wq1, wq3  (q1 vs q3, 合計1に正規化)
#   【Vスコア内部重み】 wv1, wv2, wv3  (合計1に正規化)
#   【閾値】 threshold
#
# 目的関数:
#   シグナルが出た日の1年後リターン平均（= 期待値最大化）
#   または Sharpe比（= リスク調整後パフォーマンス最大化）
#   「最低シグナル数」を設けてオーバーフィット防止

MIN_SIGNALS = 5   # 各銘柄で最低これだけのシグナル数を要求

def optuna_objective(trial):
    # ── QVT 合成重みの探索
    wt_raw = trial.suggest_float('wt_raw', 0.1, 0.7)
    wq_raw = trial.suggest_float('wq_raw', 0.1, 0.7)
    wv_raw = trial.suggest_float('wv_raw', 0.1, 0.7)
    total_w = wt_raw + wq_raw + wv_raw
    wt = wt_raw / total_w
    wq = wq_raw / total_w
    wv = wv_raw / total_w

    # ── Qスコア内部重みの探索 (q1 vs q3)
    wq1_raw = trial.suggest_float('wq1_raw', 0.1, 0.9)
    wq1 = wq1_raw
    wq3 = 1.0 - wq1

    # ── Vスコア内部重みの探索
    wv1_raw = trial.suggest_float('wv1_raw', 0.1, 0.7)
    wv2_raw = trial.suggest_float('wv2_raw', 0.1, 0.7)
    wv3_raw = trial.suggest_float('wv3_raw', 0.1, 0.7)
    total_v = wv1_raw + wv2_raw + wv3_raw
    wv1 = wv1_raw / total_v
    wv2 = wv2_raw / total_v
    wv3 = wv3_raw / total_v

    # ── 閾値の探索
    threshold = trial.suggest_float('threshold', 30.0, 70.0)

    # ── 全銘柄の平均シャープ比を計算
    sharpes = []
    n_signals_list = []
    for ticker, df in raw_data.items():
        pred_df = build_prediction_df(
            ticker, df,
            wt=wt, wq=wq, wv=wv,
            wq1=wq1, wq3=wq3,
            wv1=wv1, wv2=wv2, wv3=wv3,
            threshold=threshold,
        )
        sigs = pred_df[pred_df['signal']]
        n_signals_list.append(len(sigs))
        if len(sigs) < MIN_SIGNALS:
            sharpes.append(-5.0)   # シグナル不足はペナルティ
            continue
        std = sigs['return_1y'].std() + 1e-9
        sharpe = sigs['return_1y'].mean() / std
        sharpes.append(sharpe)

    return float(np.mean(sharpes))


print(f'🔍 Optuna最適化開始 ...')
print(f'   試行回数: {N_TRIALS}')
print(f'   銘柄数:   {len(raw_data)}')
print(f'   最低シグナル数: {MIN_SIGNALS}/銘柄')
print()

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(optuna_objective, n_trials=N_TRIALS, show_progress_bar=True)

print('\n✅ 最適化完了')
print(f'   最大シャープ比: {study.best_value:.4f}')

In [ ]:
# ─── セル9: 最適パラメータの整形・表示 ───────────────────────────────────────

bp = study.best_params

# 正規化
total_w = bp['wt_raw'] + bp['wq_raw'] + bp['wv_raw']
opt_wt  = bp['wt_raw'] / total_w
opt_wq  = bp['wq_raw'] / total_w
opt_wv  = bp['wv_raw'] / total_w

opt_wq1 = bp['wq1_raw']
opt_wq3 = 1.0 - opt_wq1

total_v = bp['wv1_raw'] + bp['wv2_raw'] + bp['wv3_raw']
opt_wv1 = bp['wv1_raw'] / total_v
opt_wv2 = bp['wv2_raw'] / total_v
opt_wv3 = bp['wv3_raw'] / total_v

opt_thr = bp['threshold']

print('=' * 55)
print('📌 最適パラメータ')
print('=' * 55)
print(f'【QVT合成重み】')
print(f'  タイミングスコア (T) : {opt_wt:.3f}  (デフォルト: 0.350)')
print(f'  クオリティスコア (Q) : {opt_wq:.3f}  (デフォルト: 0.350)')
print(f'  バリュースコア   (V) : {opt_wv:.3f}  (デフォルト: 0.300)')
print()
print(f'【Qスコア内部重み】')
print(f'  q1 (定性指標) : {opt_wq1:.3f}  (デフォルト: 0.500)')
print(f'  q3 (定量指標) : {opt_wq3:.3f}  (デフォルト: 0.500)')
print()
print(f'【Vスコア内部重み】')
print(f'  v1 (PBR系)    : {opt_wv1:.3f}  (デフォルト: 0.450)')
print(f'  v2 (PER系)    : {opt_wv2:.3f}  (デフォルト: 0.300)')
print(f'  v3 (配当・EV) : {opt_wv3:.3f}  (デフォルト: 0.250)')
print()
print(f'【閾値】')
print(f'  threshold : {opt_thr:.1f}  (デフォルト: {QVT_THRESHOLD})')
print(f'【目的関数値（シャープ比平均）】')
print(f'  {study.best_value:.4f}')
print('=' * 55)

In [ ]:
# ─── セル10: 最適重みでの予測率再計算 & デフォルトとの比較 ───────────────────

print('📊 最適パラメータで予測率を再計算中...')

opt_preds = {}
compare_rows = []

for ticker, df in raw_data.items():
    # デフォルト
    pred_def = default_preds[ticker]
    m_def = calc_accuracy(pred_def, QVT_THRESHOLD)

    # 最適
    pred_opt = build_prediction_df(
        ticker, df,
        wt=opt_wt, wq=opt_wq, wv=opt_wv,
        wq1=opt_wq1, wq3=opt_wq3,
        wv1=opt_wv1, wv2=opt_wv2, wv3=opt_wv3,
        threshold=opt_thr,
    )
    opt_preds[ticker] = pred_opt
    m_opt = calc_accuracy(pred_opt, opt_thr)

    compare_rows.append({
        'ticker':             ticker,
        'デフォルト_シグナル数': m_def['n_signal'],
        'デフォルト_予測率':     f'{m_def["accuracy"]*100:.1f}%',
        'デフォルト_平均Ret':    f'{m_def["avg_return"]*100:.1f}%',
        '最適_シグナル数':       m_opt['n_signal'],
        '最適_予測率':           f'{m_opt["accuracy"]*100:.1f}%',
        '最適_平均Ret':          f'{m_opt["avg_return"]*100:.1f}%',
    })

print()
display(pd.DataFrame(compare_rows))

In [ ]:
# ─── セル11: パラメータ重要度チャート ────────────────────────────────────────

try:
    importance = optuna.importance.get_param_importances(study)

    # 重要度の高い順にソート
    keys   = list(importance.keys())
    values = list(importance.values())

    # わかりやすいラベルに変換
    label_map = {
        'wt_raw':    'T重み（タイミング）',
        'wq_raw':    'Q重み（クオリティ）',
        'wv_raw':    'V重み（バリュー）',
        'wq1_raw':   'Q内: q1重み',
        'wv1_raw':   'V内: v1重み',
        'wv2_raw':   'V内: v2重み',
        'wv3_raw':   'V内: v3重み',
        'threshold': '閾値',
    }
    labels = [label_map.get(k, k) for k in keys]

    fig_imp = go.Figure(go.Bar(
        x=values, y=labels,
        orientation='h',
        marker_color='#3b82f6',
        text=[f'{v:.3f}' for v in values],
        textposition='outside',
    ))
    fig_imp.update_layout(
        **THEME, height=380,
        title=dict(text='🔍 パラメータ重要度（Optuna fANOVA）',
                   font=dict(size=15, color='#f1f5f9')),
        xaxis_title='重要度',
        margin=dict(l=180, r=60, t=50, b=0),
    )
    fig_imp.update_xaxes(gridcolor='#1e293b')
    fig_imp.show()
except Exception as e:
    print(f'⚠️ 重要度計算エラー: {e}')

In [ ]:
# ─── セル12: 最適化の収束プロット ────────────────────────────────────────────

trial_nums  = [t.number for t in study.trials if t.value is not None]
trial_vals  = [t.value  for t in study.trials if t.value is not None]
best_so_far = pd.Series(trial_vals).cummax().tolist()

fig_conv = go.Figure()
fig_conv.add_trace(go.Scatter(
    x=trial_nums, y=trial_vals,
    mode='markers', name='各試行',
    marker=dict(color='#6b7280', size=5, opacity=0.6),
))
fig_conv.add_trace(go.Scatter(
    x=trial_nums, y=best_so_far,
    mode='lines', name='最良値（累積）',
    line=dict(color='#10b981', width=2),
))
fig_conv.update_layout(
    **THEME, height=340,
    title=dict(text='📉 Optuna 最適化の収束',
               font=dict(size=15, color='#f1f5f9')),
    xaxis_title='試行回数',
    yaxis_title='シャープ比（平均）',
    legend=dict(bgcolor='rgba(0,0,0,0)'),
    margin=dict(l=0, r=0, t=50, b=0),
)
fig_conv.update_xaxes(gridcolor='#1e293b')
fig_conv.update_yaxes(gridcolor='#1e293b', zeroline=True, zerolinecolor='#334155')
fig_conv.show()

In [ ]:
# ─── セル13: インタラクティブ重みチューニング ─────────────────────────────────
# 最適値を初期値として表示。スライダーで手動調整して予測率をリアルタイム確認

style_w  = {'description_width': '200px'}
layout_w = widgets.Layout(width='460px')

def make_slider(desc, val, mn=0.0, mx=1.0, step=0.05):
    return widgets.FloatSlider(
        value=round(val, 3), min=mn, max=mx, step=step,
        description=desc, style=style_w, layout=layout_w,
        readout_format='.3f',
    )

s_wt  = make_slider('⏱ T重み（タイミング）', opt_wt)
s_wq  = make_slider('🏢 Q重み（クオリティ）', opt_wq)
s_wv  = make_slider('💰 V重み（バリュー）',   opt_wv)
s_wq1 = make_slider('Q: q1重み',              opt_wq1)
s_wv1 = make_slider('V: v1重み',              opt_wv1)
s_wv2 = make_slider('V: v2重み',              opt_wv2)
s_wv3 = make_slider('V: v3重み',              opt_wv3)
s_thr = make_slider('🎯 閾値',                opt_thr, mn=20.0, mx=80.0, step=1.0)

btn13 = widgets.Button(
    description='▶ 予測率を計算',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px'),
)
out13 = widgets.Output()

display(widgets.VBox([
    widgets.HTML('<h3 style="color:#60a5fa">🎛️ 重みチューニング</h3>'),
    widgets.HTML('<b>QVT合成重み（自動正規化）</b>'),
    s_wt, s_wq, s_wv,
    widgets.HTML('<b>Qスコア内部重み</b>'),
    s_wq1,
    widgets.HTML('<b>Vスコア内部重み（自動正規化）</b>'),
    s_wv1, s_wv2, s_wv3,
    widgets.HTML('<b>買いシグナル閾値</b>'),
    s_thr,
    btn13,
]), out13)

def on_calc(b):
    with out13:
        clear_output(wait=True)
        wt_v  = s_wt.value;  wq_v = s_wq.value;  wv_v = s_wv.value
        tw    = wt_v + wq_v + wv_v
        if tw < 0.01:
            display(HTML('<div style="color:#ef4444">⚠️ 重みの合計が0です</div>'))
            return
        wt_n, wq_n, wv_n = wt_v/tw, wq_v/tw, wv_v/tw
        wq1_n = s_wq1.value;  wq3_n = 1.0 - wq1_n
        tv = s_wv1.value + s_wv2.value + s_wv3.value
        if tv < 0.01:
            display(HTML('<div style="color:#ef4444">⚠️ V内部重みの合計が0です</div>'))
            return
        wv1_n, wv2_n, wv3_n = s_wv1.value/tv, s_wv2.value/tv, s_wv3.value/tv
        thr_v = s_thr.value

        rows = []
        for ticker, df in raw_data.items():
            pred_df = build_prediction_df(
                ticker, df,
                wt=wt_n, wq=wq_n, wv=wv_n,
                wq1=wq1_n, wq3=wq3_n,
                wv1=wv1_n, wv2=wv2_n, wv3=wv3_n,
                threshold=thr_v,
            )
            m = calc_accuracy(pred_df, thr_v)
            rows.append({
                'ticker':   ticker,
                'シグナル数':  m['n_signal'],
                '予測率':    f'{m["accuracy"]*100:.1f}%',
                '平均1年Ret': f'{m["avg_return"]*100:.1f}%',
                'シャープ比': f'{m["sharpe"]:.3f}',
            })
        display(pd.DataFrame(rows))
        display(HTML(
            f'<div style="color:#94a3b8; font-size:12px; margin-top:8px">'
            f'T={wt_n:.3f} / Q={wq_n:.3f} / V={wv_n:.3f} | '
            f'q1={wq1_n:.3f} | v1={wv1_n:.3f},v2={wv2_n:.3f},v3={wv3_n:.3f} | '
            f'閾値={thr_v:.1f}'
            f'</div>'
        ))

btn13.on_click(on_calc)
print('✅ 準備完了 — スライダーを調整して「▶ 予測率を計算」を押してください')

In [ ]:
# ─── セル14: 最適パラメータのPythonコード出力 ─────────────────────────────────
# checkSIGNALsへの組み込み用コードを自動生成

print('# ── checkSIGNALs 組み込み用 最適パラメータ ──')
print(f'''
# QVT 合成重み（正規化済み）
W_TIMING     = {opt_wt:.4f}   # T: タイミングスコア重み
W_QUALITY    = {opt_wq:.4f}   # Q: クオリティスコア重み
W_VALUATION  = {opt_wv:.4f}   # V: バリュースコア重み

# Qスコア内部重み
W_Q1         = {opt_wq1:.4f}  # q1（定性指標）
W_Q3         = {opt_wq3:.4f}  # q3（定量指標）

# Vスコア内部重み（正規化済み）
W_V1         = {opt_wv1:.4f}  # v1（PBR系）
W_V2         = {opt_wv2:.4f}  # v2（PER系）
W_V3         = {opt_wv3:.4f}  # v3（配当・EV系）

# 買いシグナル閾値（0-100スケール）
QVT_THRESHOLD = {opt_thr:.1f}


def calc_q_score(q1, q3):
    return W_Q1 * q1 + W_Q3 * q3


def calc_v_score(v1, v2, v3):
    return W_V1 * v1 + W_V2 * v2 + W_V3 * v3


def calc_qvt_score(t_score, q_score, v_score):
    return W_TIMING * t_score + W_QUALITY * q_score + W_VALUATION * v_score


def is_buy_signal(qvt_score):
    return qvt_score >= QVT_THRESHOLD
''')